# Time to bring out the scary stuff, let's talk quaternions

Rotation is a **super** important aspect of Aerospace as well as Spaceshot. We care a TON about how the rocket is oriented for the overall safety and tilt angle of the rocket. This is especially important for our *two stage* rockets in which we have a safety precaution where if our rocket is currently oriented pointing sideways or down in flight, to NOT ignite the second motor. Thus, our attitude estimation algorithms are super important for our rocket safety! 

### How you've seen rotation before

Typically you've learned about roll, pitch and yaw angles to describe the orientation of a object. Consider: 

![Alternative text](..\static\rotations.gif)

This can be described by a **Rotation Matrix** which simply is a matrix that describes the transformation needed to apply a specific rotation to an object's orientation. This is defined by: 

$R(\psi, \theta, \phi) = \begin{bmatrix}
\cos\psi\cos\theta & \cos\psi\sin\theta\sin\phi - \sin\psi\cos\phi & \cos\psi\sin\theta\cos\phi + \sin\psi\sin\phi \\
\sin\psi\cos\theta & \sin\psi\sin\theta\sin\phi + \cos\psi\cos\phi & \sin\psi\sin\theta\cos\phi - \cos\psi\sin\phi \\
-\sin\theta & \cos\theta\sin\phi & \cos\theta\cos\phi
\end{bmatrix}$

Such that, 

$\psi = yaw$, 
$\theta = pitch$, 
$\phi = roll$

As an example, consider a rotation of 180 degrees about the roll-axis. It's rotation matrix is defined as: 

$R = \begin{bmatrix} 1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{bmatrix}$ and  $v \in \mathbb{R}^3$

such that: 

$v_{rotated} = Rv_{initial}$

The matrix $R$ simply flips our object to now be oriented 180 degrees about the roll-axis. This is amazing! However theres two problems: 

**Problem 1:** 

Consider this matrix 

$R = \begin{bmatrix} 
-0.103135 & -0.672583 & 0.732800 \\ 
0.963479 & -0.231264 & -0.134107 \\ 
0.247547 & 0.692410 & 0.677846 
\end{bmatrix}$

As you can tell...it's super hard to tell what rotation is happening when looking at the rotation matrix ($R$) alone. This is a drawback of the **Euler** (roll, pitch, yaw) system that a rotation matrix constraints itself to.

**Problem 2:**

In mathematics we call a singularity a specific point where a function, equation, or geometric surface is not defined, breaks down, or fails to be well-behaved. This is present in the Euler system in the form of **GIMBAL LOCK**. Consider this animation: 

![Alternative text](..\static\Gimbal_Lock_Plane.gif)

In the vertical orientation of the jet the system **loses a degree of freedom**. Although physically we know the jet does not actually lose this degree of freedom, the **mathematical** Euler model that describes the orientation of the jet has this singularity. It is important that we understand this mathematical limitation.

### Your Turn

Let's build a rotation matrix from a set of rotations! Imagine our rocket starts in its original orientation and undergoes the following rotations:

Yaw: $30^\circ$
Pitch: $20^\circ$
Roll: $45^\circ$

**Your task:**

Create the rotation matrix for each individual rotation:

$$
R_z(\psi), \qquad R_y(\theta), \qquad R_x(\phi)
$$

Substitute:

$$
\psi = 30^\circ, \qquad
\theta = 20^\circ, \qquad
\phi = 45^\circ
$$

Combine the three rotations to form the overall rotation matrix:

$$
R = R_z(\psi)R_y(\theta)R_x(\phi)
$$

Calculate the final numerical value of $R$.


In [ ]:
#% exercise
#% checker: check_ex1
import numpy as np

yaw = np.deg2rad(30)
pitch = np.deg2rad(20)
roll = np.deg2rad(45)

### BEGIN STUB
R_x = None
R_y = None
R_z = None

R = None
### END STUB
### BEGIN SOLUTION
R_z = np.array([
    [np.cos(yaw), -np.sin(yaw), 0],
    [np.sin(yaw),  np.cos(yaw), 0],
    [0,            0,           1]
])
R_y = np.array([
    [ np.cos(pitch), 0, np.sin(pitch)],
    [ 0,             1, 0            ],
    [-np.sin(pitch), 0, np.cos(pitch)]
])
R_x = np.array([
    [1, 0,           0          ],
    [0, np.cos(roll), -np.sin(roll)],
    [0, np.sin(roll),  np.cos(roll)]
])
R = R_z @ R_y @ R_x
### END SOLUTION

print(f"Rotation matrix R:\n{R}")



# We solve these issues with quaternions!!! 

We define a quaternion as a mathematical representation of a rotation in 3D space. Instead of using 3 angles like roll, pitch, and yaw, a quaternion uses 4 values:

$
\boxed{
\mathbf{q}=
\begin{bmatrix}
w\\
x\\
y\\
z
\end{bmatrix}}
$

where $w$ is the scalar component and $x,y,z$ are the vector components. Thus, a quaternion can also be written as:

$
q=w+xi+yj+zk
$

A quaternion represents a rotation using an axis and an angle. **Simply put, we are describing a rotation by rotating a object by an angle $\theta$ about the unit axis $\hat{u}$ such that the quaternion is:**

$
q = 
\begin{bmatrix}
\cos(\theta/2)\\
u_x\sin(\theta/2)\\
u_y\sin(\theta/2)\\
u_z\sin(\theta/2)
\end{bmatrix} = \cos(\theta/2) + \sin(\theta/2) \hat{u} 
$

For example, a $180^\circ$ rotation about the roll ($x$) axis gives:

$
\boxed{
\begin{bmatrix}
0\
1\
0\
0
\end{bmatrix}}
$


Here is an animation to conceptualize this: 

<video width="640" height="360" controls>
  <source src="../static/QuaternionRotationCube.mp4" type="video/mp4">
  Your browser does not support the video tag.
</video>


We define a quaternion in python as this: 



In [ ]:
import quaternion as q 

quat = q.quaternion(0, 1, 0, 0)

But how do we undo a quaternion? The conjugate! Formally defined as 

$q^{-1}$

Finally, to apply a rotation of a quaternion to a vector: 

$ v_{rotated} = q \times v_{initial} \times q^{-1}$

Recall, the quaternion rotation for about the x-axis of 180 degrees is $q = (0,1,0,0)$. Intuitively for a vector $v$ pointing in the y-axis a 180 degree rotation about the x axis will flip the y-axis. Lets see if this works with quaternions: 

In [ ]:
vector =  np.array([0, 1, 0])
vector_rotated = quat * q.quaternion(0, *vector) * quat.conjugate() 
vector_rotated = np.array([vector_rotated.x, vector_rotated.y, vector_rotated.z])
print(vector_rotated)


In summary:  we convert the vector to a quaternion (needed for python...so w = 0, x,y,z = vector), rotate it, and convert it back to a vector. 

Say we wanted to do two rotations...one of 180 degrees about the x axis as well as 180 degrees about the z axis. Intuitively this would return the vector back to its original position. We can simply combine rotations by using *quaternion multiplication*. Let 1 be our original orientation, 2 be our second orientation after the first rotation and 3 be our final orientation. We define *quaternion multiplication* as: 

$q_3 = q^3_2 \times q^2_1$

Lets test this: 

In [ ]:
vector =  np.array([0, 1, 0])
quat_23 = q.quaternion(0, 1, 0, 0)
quat_12 = q.quaternion(0, 0, 0, 1)
quat = quat_23 * quat_12
vector_rotated = quat * q.quaternion(0, *vector) * quat.conjugate() 
vector_rotated = np.array([vector_rotated.x, vector_rotated.y, vector_rotated.z])
print(vector_rotated)

### It matches! 

**One last advantage of quaternions: the double rotation.**

For any rotation, there are exactly two quaternions that represent it: **q** and **−q**. This isn't true for rotation matrices!! Negating all four components (w, x, y, z) → (−w, −x, −y, −z) doesn't change the orientation it encodes.

Why? A rotation by angle θ about axis **n̂** is encoded as:

$$q = \left(\cos\frac{\theta}{2},\ \sin\frac{\theta}{2}\,\hat{n}\right) = -q$$

Negating q is equivalent to replacing θ with θ + 360°, and rotating by θ or by θ + 360° about the same axis lands you in the exact same orientation. So q and −q trace out different paths (one spins "the long way around"), but they arrive at the same result. By convention however, we utilize $\omega$ to be always positive. 


### Your Turn

Let's build a rotation *quaternion* from a set of rotations! Imagine our rocket starts in its original orientation and undergoes the following rotations:

Yaw: $30^\circ$
Pitch: $20^\circ$
Roll: $45^\circ$

**Your task:**

Create the quaternion for each individual rotation:

$$
q_z(\psi), \qquad q_y(\theta), \qquad q_x(\phi)
$$

where a rotation by angle $\alpha$ about a unit axis $\hat{n}$ is:

$$
q(\alpha) = \left(\cos\frac{\alpha}{2},\ \sin\frac{\alpha}{2}\,\hat{n}\right)
$$

Substitute:

$$
\psi = 30^\circ, \qquad
\theta = 20^\circ, \qquad
\phi = 45^\circ
$$

Combine the three rotations to form the overall rotation quaternion, using quaternion multiplication (not addition!):

$$
q = q_z(\psi)\, q_y(\theta)\, q_x(\phi)
$$

Calculate the final numerical value of $q = (w, x, y, z)$, and verify it's a unit quaternion ($w^2+x^2+y^2+z^2=1$).

In [ ]:
#% exercise
#% checker: check_ex2
import numpy as np
import quaternion as q

yaw = np.deg2rad(30)
pitch = np.deg2rad(20)
roll = np.deg2rad(45)

### BEGIN STUB
q_z = None
q_y = None
q_x = None

quat = None
### END STUB
### BEGIN SOLUTION
q_z = q.quaternion(np.cos(yaw / 2), 0, 0, np.sin(yaw / 2))
q_y = q.quaternion(np.cos(pitch / 2), 0, np.sin(pitch / 2), 0)
q_x = q.quaternion(np.cos(roll / 2), np.sin(roll / 2), 0, 0)

quat = q_z * q_y * q_x
### END SOLUTION

print(f"Rotation quaternion q:\n{quat}")


# So why does all of this matter? 

In this next module you'll see where we utilize quaternions: the **Multiplicative Extended Kalman Filter (MEKF)**. This is simply the background you need. Any career in GNC will require you to understand quaternions....trust me I've spent hours reading literature on quaternions ([yes dual quaternions are a thing if you're looking for a nice bed time story](https://en.wikipedia.org/wiki/Dual_quaternion)😉). 